# 08 - LLM + RAG Evaluation

Evaluates the LLM+RAG routing classifier on the same frozen `test.csv`, using the routing
index built in `07_rag_index_creation.ipynb` (train.csv only). Retrieval similarity and
retrieved document IDs are saved alongside the prediction as diagnostic RAG information --
never combined with BERT softmax or the LLM's own output into one confidence value.

**Validation-only side pass:** as in `06_llm_evaluation.ipynb`, this notebook first runs the
classifier over the *validation* split, purely to get a validation macro F1 comparable to
BERT's and the plain LLM's. Once all three methods' validation macro F1 are known, this
notebook resolves the demo's default routing method
(`configs/base.yaml: demo.default_routing_method`, docs/BLUEPRINT.md Section 10) --
whichever method scores best, not automatically LLM+RAG.

All logic lives in `src/newstart_ai/rag/` and `src/newstart_ai/models/llm/`.

### Load the frozen split and assemble the LLM+RAG classifier

**Purpose:** Load all three splits, then build the three pieces that together make up the
LLM+RAG method: a `Retriever` over the index built in notebook 07, a `GeminiProvider` (the
same LLM used in notebook 06), and the RAG-specific prompt template that has room for
retrieved context. `RagEnhancedClassifier` combines all three into one object with a single
`classify()` method.

**Why this step is necessary:** This mirrors notebook 06's setup almost exactly, on purpose
-- the only real difference between the plain LLM and the LLM+RAG method is that this
classifier retrieves similar training examples first and includes them in the prompt. Using
the same underlying `GeminiProvider` and the same base model keeps the comparison isolated
to "does adding retrieved context help?" rather than mixing in an unrelated variable.

**Inputs:** `data/splits/*.csv`, the routing index built in notebook 07,
`configs/rag.yaml`/`configs/llm.yaml`, and `prompts/rag_classification/v1.yaml`.

**Output:** `retriever`, `llm_provider`, `prompt`, and `rag_classifier` (the combined
object used for every prediction below).

**How to interpret the result:** The printed line confirms the embedding model, how many
similar examples are retrieved per query (`top_k`), and which prompt version is in use.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

from newstart_ai.config import load_settings
from newstart_ai.data import load_split
from newstart_ai.models.llm import GeminiProvider, load_rag_classification_prompt
from newstart_ai.rag import Retriever, RagEnhancedClassifier
from newstart_ai.evaluation import evaluate_predictions, save_predictions, save_metrics_report

settings = load_settings()
train_df, val_df, test_df, manifest = load_split(settings)
ds_cfg = settings.base.dataset

# Retriever reads from the index notebook 07 built (train_df only) -- it is never rebuilt
# or modified here, only queried.
retriever = Retriever(settings)
# Same GeminiProvider as notebook 06 -- the model itself is unchanged; only the prompt and
# the retrieved context differ between the plain-LLM and LLM+RAG methods.
llm_provider = GeminiProvider(settings)
prompt = load_rag_classification_prompt(settings)
# RagEnhancedClassifier ties retrieval and generation together: for each document it
# retrieves similar training examples, then asks the LLM to classify using them as context.
rag_classifier = RagEnhancedClassifier(retriever, llm_provider, prompt)
print(f"embedding model: {settings.rag.embedding_model}   top_k: {settings.rag.top_k}   prompt: {prompt.version}")

embedding model: gemini-embedding-001   top_k: 5   prompt: v1


### Define the "classify every row with RAG" helper

**Purpose:** Define `classify_split()`, the LLM+RAG equivalent of notebook 06's helper --
used identically for the validation pass and the frozen test pass below.

**Why this step is necessary:** Same reasoning as notebook 06: one shared implementation for
both passes, and a failed call is recorded as an explicit error rather than silently
skipped or faked.

**Inputs:** A DataFrame (`val_df` or `test_df`), the split name, and the `rag_classifier`
built above.

**Output:** A function object; no output until it's called below.

**How to interpret the result:** Nothing to interpret yet -- this defines the tool the next
cells use.

In [2]:
def classify_split(df, split_name: str, method: str = "llm_rag"):
    """Classifies every row in df with the LLM+RAG classifier. A failed call is recorded
    in `errors` and excluded from results/metrics rather than silently faking a label."""
    results = []
    errors = []
    texts = df[ds_cfg.text_column].tolist()
    doc_ids = df[ds_cfg.id_column].astype(str).tolist()
    true_labels = df[ds_cfg.label_column].tolist()

    for doc_id, text, true_label in zip(doc_ids, texts, true_labels):
        try:
            # rag_classifier.classify() retrieves similar training examples, builds the
            # RAG prompt around them, and asks the LLM to classify -- all in one call.
            result = rag_classifier.classify(
                text, document_id=doc_id, method=method, extra_metadata={"split": split_name}
            )
            result.true_label = true_label
            results.append(result)
        except Exception as exc:
            errors.append({"document_id": doc_id, "error": str(exc)})

    print(f"{split_name}: {len(results)} succeeded, {len(errors)} failed")
    return results, errors

## Validation-only side pass

### Validation-only pass: classify the validation split with LLM+RAG

**Purpose:** Run every validation document through the LLM+RAG classifier.

**Why this step is necessary:** Just like notebook 06's validation pass, this produces a
validation macro F1 for LLM+RAG that's directly comparable to BERT's and the plain LLM's --
the three-way comparison that resolves the demo's default routing method a few cells below.
This pass never touches `test_df`.

**Inputs:** `val_df` (121 documents).

**Output:** `val_results` and `val_errors`.

**How to interpret the result:** An empty `val_errors` list (as seen in this run) means
every validation document got a real prediction to score in the next cell.

In [3]:
val_results, val_errors = classify_split(val_df, split_name="validation")
val_errors

validation: 121 succeeded, 0 failed


[]

### Score the validation-only pass

**Purpose:** Compute LLM+RAG's validation metrics and save them, exactly as notebook 06 did
for the plain LLM.

**Why this step is necessary:** This saved report (`llm_rag_validation_metrics.json`) is one
of the three inputs the routing-method decision below reads back -- alongside BERT's and the
plain LLM's validation macro F1.

**Inputs:** `val_results`.

**Output:** `val_report`, saved to `artifacts/reports/llm_rag_validation_metrics.json`.

**How to interpret the result:** The printed validation macro F1 for LLM+RAG is compared,
a few cells down, directly against BERT's and the LLM's validation macro F1.

In [4]:
val_report = evaluate_predictions(
    true_labels=[r.true_label for r in val_results],
    predicted_labels=[r.predicted_label for r in val_results],
    label_order=settings.base.labels,
    method="llm_rag",
    split="validation",
    latencies_ms=[r.latency_ms for r in val_results],
    total_token_usage=sum(r.token_usage.total_tokens for r in val_results if r.token_usage),
    total_estimated_cost=sum(r.estimated_cost for r in val_results if r.estimated_cost is not None),
)
save_predictions(val_results, method="llm_rag", split="validation", settings=settings)
save_metrics_report(val_report, settings)
print(f"LLM+RAG validation macro F1: {val_report.macro_f1:.4f}")

LLM+RAG validation macro F1: 1.0000


## Frozen test-set evaluation

`test.csv` is touched here, exactly once, for the LLM+RAG method.

### Frozen test-set pass: classify the test split with LLM+RAG (touched here, exactly once)

**Purpose:** Run every test-set document through the LLM+RAG classifier -- the method's one
and only frozen evaluation.

**Why this step is necessary:** This is the number reported as LLM+RAG's real performance,
directly comparable to BERT's (notebook 05) and the plain LLM's (notebook 06) test-set
results, since all three methods are scored on the identical 151 test documents.

**Inputs:** `test_df` (151 documents).

**Output:** `test_results` and `test_errors`.

**How to interpret the result:** An empty `test_errors` list means every test document
received a real prediction from the LLM+RAG classifier.

In [5]:
test_results, test_errors = classify_split(test_df, split_name="test")
test_errors

test: 151 succeeded, 0 failed


[]

### Save row-level test predictions before scoring

**Purpose:** Write every individual LLM+RAG test prediction to
`artifacts/predictions/llm_rag_test.json`, before computing any summary metric.

**Why this step is necessary:** Same reasoning as every earlier evaluation notebook: raw,
per-document evidence is preserved on disk so the error analysis in notebook 09 can compare
LLM+RAG's individual predictions against BERT's and the plain LLM's, document by document.

**Inputs:** `test_results`.

**Output:** A JSON file on disk.

**How to interpret the result:** No output is printed; this is a side effect later
notebooks depend on.

In [6]:
save_predictions(test_results, method="llm_rag", split="test", settings=settings)

WindowsPath('D:/USD/Projects/a590/newstart-ai/newstart_ai_benchmark/artifacts/predictions/llm_rag_test.json')

### Score the frozen test-set pass

**Purpose:** Compute LLM+RAG's headline test metrics -- accuracy, macro/weighted F1,
per-class metrics, confusion matrix, latency, token usage, and cost -- from the 151 test
predictions.

**Why this step is necessary:** This is LLM+RAG's final, reportable result, saved in the
same shape as BERT's and the LLM's so notebook 09 can put all three side by side without any
special-casing.

**Inputs:** `test_results`.

**Output:** `test_report`, saved to `artifacts/reports/llm_rag_test_metrics.json`.

**How to interpret the result:** Compare `test_report.macro_f1` here against notebook 06's
LLM result -- in this project's run they came out numerically identical, which notebook 09
investigates further (the RAG step retrieved similar examples but did not change a single
predicted label on this test set).

In [7]:
test_report = evaluate_predictions(
    true_labels=[r.true_label for r in test_results],
    predicted_labels=[r.predicted_label for r in test_results],
    label_order=settings.base.labels,
    method="llm_rag",
    split="test",
    latencies_ms=[r.latency_ms for r in test_results],
    total_token_usage=sum(r.token_usage.total_tokens for r in test_results if r.token_usage),
    total_estimated_cost=sum(r.estimated_cost for r in test_results if r.estimated_cost is not None),
    notes=[
        "IRS test slice is very small (~4-5 documents); IRS per-class metrics are "
        "statistically noisy and should be reported as uncertain, not precise."
    ],
)
save_metrics_report(test_report, settings)
test_report.model_dump()

{'method': 'llm_rag',
 'split': 'test',
 'accuracy': 0.9867549668874173,
 'macro_precision': 0.9535256410256411,
 'macro_recall': 0.9892045454545455,
 'macro_f1': 0.9693874078630312,
 'weighted_f1': 0.9870158453278095,
 'per_class': [{'label': 'USCIS',
   'precision': 0.9807692307692307,
   'recall': 1.0,
   'f1': 0.9902912621359223,
   'support': 51},
  {'label': 'DMV',
   'precision': 1.0,
   'recall': 0.9818181818181818,
   'f1': 0.9908256880733946,
   'support': 55},
  {'label': 'SSA',
   'precision': 1.0,
   'recall': 0.975,
   'f1': 0.9873417721518988,
   'support': 40},
  {'label': 'IRS',
   'precision': 0.8333333333333334,
   'recall': 1.0,
   'f1': 0.9090909090909091,
   'support': 5}],
 'confusion_matrix': [[51, 0, 0, 0],
  [1, 54, 0, 0],
  [0, 0, 39, 1],
  [0, 0, 0, 5]],
 'confusion_matrix_labels': ['USCIS', 'DMV', 'SSA', 'IRS'],
 'mean_latency_ms': 1555.7528648954133,
 'total_token_usage': 1170333,
 'total_estimated_cost': 0.12307290000000005,
 'cost_per_document': 0.000815

## Resolve the demo's default routing method

Compares all three methods' **validation** macro F1 (never test) and writes the winner to
`configs/base.yaml: demo.default_routing_method` -- not automatically LLM+RAG
(docs/BLUEPRINT.md Section 10).

### Gather all three methods' validation macro F1 for the routing decision

**Purpose:** Collect the validation macro F1 already computed for BERT (from its saved
artifact metadata), the plain LLM (from its saved validation report), and LLM+RAG (from
`val_report`, computed earlier in this notebook), and lay them side by side.

**Why this step is necessary:** The project design requires the demo app's default routing
method to be chosen by comparing all three methods' validation performance -- never assumed
to be LLM+RAG or any other method by default. This cell is where that comparison actually
happens, reading back results that were deliberately saved by earlier notebooks specifically
so this cross-method comparison would be possible without re-running anything.

**Inputs:** BERT's artifact metadata (`artifacts/models/<id>/metadata.json`), the LLM's
saved validation metrics report, and this notebook's own `val_report`.

**Output:** `routing_comparison`, a small DataFrame with one row per method, sorted from
best to worst validation macro F1.

**How to interpret the result:** The top row (index 0) is the method the next cell will
select as the demo's default. In this project's run all three methods tied at a validation
macro F1 of 1.0, so the method listed first (`bert`) won the tie -- the same deterministic,
documented tie-break rule used for the long-document strategy in notebook 04.

In [8]:
import yaml

from newstart_ai.models.bert import latest_ready_artifact_id, load_artifact_metadata
from newstart_ai.evaluation import load_metrics_report

# BERT's validation macro F1 was already computed and saved as part of its own checkpoint
# selection in notebook 04 -- no need to recompute it here.
bert_artifact_id = latest_ready_artifact_id(settings)
bert_metadata = load_artifact_metadata(settings, bert_artifact_id)
bert_val_macro_f1 = bert_metadata.validation_metrics["best_validation_macro_f1"]

# The plain LLM's validation macro F1 was saved by notebook 06's validation-only pass.
llm_val_report = load_metrics_report(method="llm", split="validation", settings=settings)
llm_val_macro_f1 = llm_val_report.macro_f1

# LLM+RAG's validation macro F1 was just computed a few cells above in this notebook.
llm_rag_val_macro_f1 = val_report.macro_f1

import pandas as pd

routing_comparison = pd.DataFrame(
    {
        "method": ["bert", "llm", "llm_rag"],
        "validation_macro_f1": [bert_val_macro_f1, llm_val_macro_f1, llm_rag_val_macro_f1],
    }
).sort_values("validation_macro_f1", ascending=False, ignore_index=True)
routing_comparison

,method,validation_macro_f1
0,bert,1.0
1,llm,1.0
2,llm_rag,1.0


### Write the resolved routing method back to configuration

**Purpose:** Take the winning method from `routing_comparison` and persist it to
`configs/base.yaml: demo.default_routing_method`.

**Why this step is necessary:** Exactly like the long-document strategy decision in notebook
04, this makes a research decision durable and visible in the shared config file, rather
than a fact that only exists inside this notebook's output. The future demo app will read
this value to decide which method's prediction actually routes a document to a downstream
agency, without needing any code change when the answer changes (e.g. if the dataset grows
and the ranking changes).

**Inputs:** `routing_comparison` (the sorted table above) and the existing
`configs/base.yaml` file.

**Output:** An updated `configs/base.yaml` file.

**How to interpret the result:** After this cell, `demo.default_routing_method` is no
longer `null` -- it holds the method that won the validation comparison (`bert`, in this
project's run), and every notebook that loads settings afterward will see that value.

In [9]:
winning_method = routing_comparison.iloc[0]["method"]
print(f"Default demo routing method: {winning_method}")

base_yaml_path = settings.project_root / "configs" / "base.yaml"
with open(base_yaml_path, "r", encoding="utf-8") as f:
    raw_config = yaml.safe_load(f)

# Overwrite just this one field -- everything else in base.yaml is left untouched.
raw_config["demo"]["default_routing_method"] = winning_method

with open(base_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(raw_config, f, sort_keys=False)

print(f"configs/base.yaml demo.default_routing_method -> {winning_method}")

Default demo routing method: bert
configs/base.yaml demo.default_routing_method -> bert


## Summary for the next notebook

- LLM+RAG frozen test macro F1: see `test_report.macro_f1` above.
- Demo default routing method resolved from validation macro F1 (BERT vs LLM vs LLM+RAG),
  written to `configs/base.yaml`.
- Row-level predictions and metrics saved for all three methods' test-set runs.
- Next: `09_model_comparison_and_error_analysis.ipynb` builds the full head-to-head
  comparison and error analysis.